In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M25.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7686159408834409, 'n_it': 0.39529601714472695}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 700

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.QuasirandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[13.76234856834038, 13.250243835242673, 13.981132683306502, 16.380961986525936, 14.470408388750052, 13.545730838011956, 17.010550834466184, 13.890188843515798, 13.696271401203305, 15.061526718734566, 16.902024437523792, 13.467273191655513, 13.493138803344745, 17.047437050674368, 14.891206569597736, 17.708418560345777, 18.092121539982426, 17.459199726505577, 13.5830302212222, 14.910470114614407, 13.682874066595499, 15.419356735005984, 14.717308021241365, 14.699375951964088, 14.725377420031474, 13.705285229812576, 13.572902872480809, 15.546285376071836, 13.665937805380722, 13.355878643754343, 18.20063966517931, 13.727025256993146, 13.34022752963122, 13.3227943063325, 16.912934567166747, 14.7794272986188, 13.550179413195854, 14.263479357963323, 13.708110621375248, 14.792874192865408, 13.765687606488852, 15.274024196591252, 13.777785956679681, 15.049034271755865, 13.908508335783361, 13.807888305120168, 16.71102166967735, 14.518228386909325, 14.316250180687716, 14.627200210868626, 13.468740

In [5]:
np.average(y_max_arr)

np.float64(15.026606989650178)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M25/DataGenerated/quasirandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)